In [5]:
from pathlib import Path
from collections import deque

import cv2
import numpy as np
import torch
from torch import nn
from torchvision import transforms
import matplotlib.pyplot as plt
from PIL import Image

# ── Rutas ───────────────────────────────────────────────────
RUTA_VIDEO = Path("C:/Users/USER/Documents/DataScience/01-python-intro/videos/14951336_2160_3840_30fps.mp4")
RUTA_MODELO = Path("C:/Users/USER/Documents/DataScience/models/cnn_propia_cifar10.pt")
RUTA_SALIDA = RUTA_VIDEO.with_name("perro_clasificado.avi")    # se guarda junto al video original

NOMBRES = ["avion", "auto", "pajaro", "gato", "venado",
           "perro", "rana", "caballo", "barco", "camion"]      # sin tildes: OpenCV no las dibuja
ANCHO_SALIDA = 540                                             # reduce el 4K para que sea rápido

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ── Arquitectura (idéntica al ejercicio 5) ──────────────────
class CNNPropia(nn.Module):
    def __init__(self, num_clases=10):
        super().__init__()
        self.extractor = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.clasificador = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_clases),
        )

    def forward(self, x):
        return self.clasificador(self.extractor(x))


# ── Cargar el modelo entrenado ──────────────────────────────
modelo = CNNPropia()
modelo.load_state_dict(torch.load(RUTA_MODELO, map_location=DEVICE))
modelo = modelo.to(DEVICE).eval()

transform_cnn = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])


@torch.no_grad()
def probabilidades_frame(frame_bgr):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)          # OpenCV usa BGR
    imagen = Image.fromarray(rgb).resize((32, 32))             # tamaño CIFAR-10
    tensor = transform_cnn(imagen).unsqueeze(0).to(DEVICE)
    return torch.softmax(modelo(tensor), dim=1)[0].cpu()


# ── Procesar el video ───────────────────────────────────────
cap = cv2.VideoCapture(str(RUTA_VIDEO))
assert cap.isOpened(), "No se pudo abrir el video: revisa el nombre o la ruta"

fps = cap.get(cv2.CAP_PROP_FPS) or 30
ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
alto_salida = int(alto * ANCHO_SALIDA / ancho)                 # mantiene la proporción vertical
salida = cv2.VideoWriter(str(RUTA_SALIDA), cv2.VideoWriter_fourcc(*"mp4v"),
                         fps, (ANCHO_SALIDA, alto_salida))

historial = deque(maxlen=10)        # suaviza la etiqueta entre frames